In [1]:
import networkx as nx
import pandas as pd

# Load STRING network
string_data = pd.read_csv("/projects/ksamart@xsede.org/integrative-drugrep-tb/data/metadata/STRING_hs_interactions.txt", sep="\t", names=['gene1','gene2','weight'],header=0)
string_data

,gene1,gene2,weight
0,ARF5,DYRK4,0.166000
1,ARF5,PPP5C,0.254968
2,ARF5,MAP4K5,0.157276
3,ARF5,RALBP1,0.156000
4,ARF5,PKP2,0.160210
...,...,...,...
1972242,ZNF788,ANKRD65,0.162811
1972243,ZNF788,PCSK5,0.215378
1972244,ZNF788,OBSCN,0.236000
1972245,ZNF788,RAB5C,0.195550


In [2]:
# Create graph (assuming 'gene1', 'gene2', 'weight' columns)
G = nx.Graph()
for _, row in string_data.iterrows():
    G.add_edge(row['gene1'], row['gene2'], weight=1-row['weight'])


In [ ]:
node_types = pd.read_csv("/projects/ksamart@xsede.org/integrative-drugrep-tb/data/v2/figure3/node_types_with_perturbed_genes.tsv",sep="\t",names=['node','type'],header=0)
# Assuming the 'type' column contains labels like 'node' and 'target'
sources = node_types[node_types['type'] == 'source']['node'].tolist()
targets = node_types[node_types['type'] == 'target']['node'].tolist()

from joblib import Parallel, delayed

def process_source(source):
    paths = {}
    if source in G:
        paths_from_source = nx.single_source_dijkstra_path(G, source, weight='weight')
        for target in targets:
            paths[(source, target)] = paths_from_source.get(target, None)
    return paths

results = Parallel(n_jobs=-1)(delayed(process_source)(src) for src in sources)
shortest_paths = {k: v for d in results for k, v in d.items()}

In [27]:
shortest_paths

{('PARP1', 'DNM1L'): ['PARP1', 'UBC', 'DNM1L'],
 ('PARP1', 'KIF20A'): ['PARP1', 'UBC', 'KIF20A'],
 ('PARP1', 'ARL4C'): ['PARP1', 'UBC', 'ARL4C'],
 ('PARP1', 'TXNDC9'): ['PARP1', 'UBC', 'TXNDC9'],
 ('PARP1', 'CDK6'): ['PARP1', 'UBC', 'CCND1', 'CDK6'],
 ('PARP1', 'CDKN1A'): ['PARP1', 'UBC', 'CDKN1A'],
 ('PARP1', 'STUB1'): ['PARP1', 'UBC', 'STUB1'],
 ('PARP1', 'IKZF1'): ['PARP1', 'UBC', 'HDAC1', 'IKZF1'],
 ('PARP1', 'HMG20B'): ['PARP1', 'UBC', 'HDAC1', 'KDM1A', 'HMG20B'],
 ('PARP1', 'LYPLA1'): ['PARP1', 'UBC', 'LYPLA1'],
 ('PARP1', 'VAT1'): ['PARP1', 'UBC', 'VAT1'],
 ('PARP1', 'CNPY3'): ['PARP1',
  'UBC',
  'EPRS',
  'MARS',
  'EEF1E1',
  'NAT9',
  'CNPY3'],
 ('PARP1', 'YME1L1'): ['PARP1', 'UBC', 'YME1L1'],
 ('PARP1', 'TCFL5'): ['PARP1', 'UBC', 'E2F1', 'TOPBP1', 'TCFL5'],
 ('PARP1', 'MTHFD2'): ['PARP1', 'UBC', 'ATIC', 'GART', 'MTHFD2'],
 ('PARP1', 'PRSS23'): ['PARP1', 'UBC', 'ELAVL1', 'PRSS23'],
 ('PARP1', 'SLC2A6'): ['PARP1', 'UBC', 'GALE', 'SLC2A6'],
 ('PARP1', 'CHEK2'): ['PARP1', 'UBC'

In [28]:
from collections import Counter

# Flatten the shortest paths
path_nodes = [node for path in shortest_paths.values() if path for node in path]
node_participation = Counter(path_nodes)

In [29]:
# Create a subgraph based on relevant shortest paths
relevant_edges = [(u, v) for path in shortest_paths.values() if path for u, v in zip(path, path[1:])]
subgraph = G.edge_subgraph(relevant_edges)

# Compute betweenness centrality
bc_scores = nx.betweenness_centrality(subgraph, weight='weight')

In [30]:
sorted_bc = sorted(bc_scores.items(), key=lambda x: x[1], reverse=True)

In [31]:
# Define BC cutoff (example: top 2.5%)
top_nodes_cutoff = int(len(sorted_bc) * 0.025)
important_nodes = [node for node, bc in sorted_bc[:top_nodes_cutoff]]

In [32]:
len(important_nodes)
important_nodes

['UBC',
 'RPS3',
 'CBL',
 'HSP90AA1',
 'RPL27A',
 'RPL5',
 'ATP5A1',
 'MRTO4',
 'ATP5B',
 'POLR2A',
 'ESR1',
 'HSPA8',
 'PCNA',
 'BCAS2',
 'CCNB1',
 'RHOA',
 'SRC',
 'RAF1',
 'TP53']

In [33]:
important_nodes_ = set(important_nodes) - set(sources) - set(targets)
print(type(shortest_paths))
# Subset dictionary
filtered_paths = {
    key: path for key, path in shortest_paths.items()
    if path is not None and any(gene in path for gene in important_nodes_)
}


<class 'dict'>


In [34]:
print(len(shortest_paths))
print(len(filtered_paths))
filtered_paths

6681
6145


{('PARP1', 'DNM1L'): ['PARP1', 'UBC', 'DNM1L'],
 ('PARP1', 'KIF20A'): ['PARP1', 'UBC', 'KIF20A'],
 ('PARP1', 'ARL4C'): ['PARP1', 'UBC', 'ARL4C'],
 ('PARP1', 'TXNDC9'): ['PARP1', 'UBC', 'TXNDC9'],
 ('PARP1', 'CDK6'): ['PARP1', 'UBC', 'CCND1', 'CDK6'],
 ('PARP1', 'CDKN1A'): ['PARP1', 'UBC', 'CDKN1A'],
 ('PARP1', 'STUB1'): ['PARP1', 'UBC', 'STUB1'],
 ('PARP1', 'IKZF1'): ['PARP1', 'UBC', 'HDAC1', 'IKZF1'],
 ('PARP1', 'HMG20B'): ['PARP1', 'UBC', 'HDAC1', 'KDM1A', 'HMG20B'],
 ('PARP1', 'LYPLA1'): ['PARP1', 'UBC', 'LYPLA1'],
 ('PARP1', 'VAT1'): ['PARP1', 'UBC', 'VAT1'],
 ('PARP1', 'CNPY3'): ['PARP1',
  'UBC',
  'EPRS',
  'MARS',
  'EEF1E1',
  'NAT9',
  'CNPY3'],
 ('PARP1', 'YME1L1'): ['PARP1', 'UBC', 'YME1L1'],
 ('PARP1', 'TCFL5'): ['PARP1', 'UBC', 'E2F1', 'TOPBP1', 'TCFL5'],
 ('PARP1', 'MTHFD2'): ['PARP1', 'UBC', 'ATIC', 'GART', 'MTHFD2'],
 ('PARP1', 'PRSS23'): ['PARP1', 'UBC', 'ELAVL1', 'PRSS23'],
 ('PARP1', 'SLC2A6'): ['PARP1', 'UBC', 'GALE', 'SLC2A6'],
 ('PARP1', 'CHEK2'): ['PARP1', 'UBC'

In [35]:
# Convert to DataFrame
# Convert to DataFrame
filtered_paths_df = pd.DataFrame([
    {"Source": key[0], "Target": key[1], "Path": ",".join(path)}
    for key, path in filtered_paths.items()
])


filtered_paths_df
# Write DataFrame to CSV
filtered_paths_df.to_csv("/projects/ksamart@xsede.org/integrative-drugrep-tb/data/v2/figure3/filtered_shortest_paths_bcscores.csv", index=False,sep='\t')

In [36]:
# Save results
pd.DataFrame(sorted_bc, columns=['Node', 'BC_Score']).to_csv("/projects/ksamart@xsede.org/integrative-drugrep-tb/data/v2/figure3/bc_scores.csv", index=False)

In [37]:
import pandas as pd

# Load STRING network
# Assuming STRING network has 'gene1', 'gene2', and 'weight' columns
string_data = pd.read_csv("/projects/ksamart@xsede.org/integrative-drugrep-tb/data/metadata/STRING_hs_interactions.txt", sep="\t", names=['gene1','gene2','weight'],header=0)

# Extract all unique genes from the shortest paths
genes_in_paths = set(
    gene for path in filtered_paths.values() if path for gene in path
)

# Subset the STRING network by filtering rows where both genes are in `genes_in_paths`
subset_network = string_data[
    (string_data['gene1'].isin(genes_in_paths)) &
    (string_data['gene2'].isin(genes_in_paths))
]

# Save the subsetted network to a CSV
subset_network.to_csv("/projects/ksamart@xsede.org/integrative-drugrep-tb/data/v2/figure3/subset_string_network_shortestpaths_bcscores.csv", index=False)

# Print a summary
print(f"Original STRING network size: {string_data.shape[0]} edges")
print(f"Subset STRING network size: {subset_network.shape[0]} edges")


Original STRING network size: 1972247 edges
Subset STRING network size: 22646 edges


In [39]:
# Compute average BC scores for each path
path_avg_bc = {}
for key, path in shortest_paths.items():
    if path:  # Ensure the path is not None
        avg_bc = sum(bc_scores.get(node, 0) for node in path) / len(path)
        path_avg_bc[key] = avg_bc

# Group paths by source node
paths_by_source = {}
for (source, target), avg_bc in path_avg_bc.items():
    if source not in paths_by_source:
        paths_by_source[source] = []
    paths_by_source[source].append(((source, target), avg_bc))

# Get top paths for each source node
top_percentage = 0.02  # Adjust this as needed
top_paths_per_source = {}

for source, paths in paths_by_source.items():
    # Sort paths by average BC score for the current source
    sorted_paths = sorted(paths, key=lambda x: x[1], reverse=True)
    num_top_paths = max(1, int(len(sorted_paths) * top_percentage))  # At least 1 path
    top_paths = sorted_paths[:num_top_paths]

    # Filter paths to include only those passing through important nodes
    filtered_paths = {
        key: shortest_paths[key]
        for key, avg_bc in top_paths
        if any(gene in shortest_paths[key] for gene in important_nodes_)
    }

    top_paths_per_source[source] = filtered_paths

# Print results for each source node
for source, paths in top_paths_per_source.items():
    print(f"Source: {source}")
    print(f"Number of top paths: {len(paths)}")
    for key, path in paths.items():
        print(f"  Target: {key[1]}, Path: {path}")
    print("-" * 30)



Source: PARP1
Number of top paths: 7
  Target: HSPA8, Path: ['PARP1', 'UBC', 'HSPA8']
  Target: PCNA, Path: ['PARP1', 'UBC', 'PCNA']
  Target: CCNB1, Path: ['PARP1', 'UBC', 'CCNB1']
  Target: RHOA, Path: ['PARP1', 'UBC', 'RHOA']
  Target: CDKN1A, Path: ['PARP1', 'UBC', 'CDKN1A']
  Target: APP, Path: ['PARP1', 'UBC', 'APP']
  Target: HMGCR, Path: ['PARP1', 'UBC', 'HMGCR']
------------------------------
Source: ADRB2
Number of top paths: 7
  Target: HSPA8, Path: ['ADRB2', 'UBC', 'HSPA8']
  Target: PCNA, Path: ['ADRB2', 'UBC', 'PCNA']
  Target: CCNB1, Path: ['ADRB2', 'UBC', 'CCNB1']
  Target: RHOA, Path: ['ADRB2', 'UBC', 'RHOA']
  Target: CDKN1A, Path: ['ADRB2', 'UBC', 'CDKN1A']
  Target: APP, Path: ['ADRB2', 'UBC', 'APP']
  Target: HMGCR, Path: ['ADRB2', 'UBC', 'HMGCR']
------------------------------
Source: ALAS1
Number of top paths: 7
  Target: HSPA8, Path: ['ALAS1', 'UBC', 'HSPA8']
  Target: PCNA, Path: ['ALAS1', 'UBC', 'PCNA']
  Target: CCNB1, Path: ['ALAS1', 'UBC', 'CCNB1']
  Target

In [40]:
# Prepare data for DataFrame
csv_rows = []
for source, paths in top_paths_per_source.items():
    for (source_node, target_node), path in paths.items():
        if path:  # Ensure the path is not None
            # Calculate average BC score
            avg_bc = sum(bc_scores.get(node, 0) for node in path) / len(path)
            # Add row to list
            csv_rows.append({
                "Source": source_node,
                "Target": target_node,
                "Path": " -> ".join(path),  # Convert path list to a string
                "Average_BC_Score": avg_bc
            })

# Convert to DataFrame
filtered_paths_df = pd.DataFrame(csv_rows)

# Write DataFrame to CSV
output_file = "/projects/ksamart@xsede.org/integrative-drugrep-tb/data/v2/figure3/filtered_shortest_paths_bcscores_persource.csv"
filtered_paths_df.to_csv(output_file, index=False, sep='\t')

# Print confirmation
print(f"Top paths per source written to: {output_file}")


Top paths per source written to: /projects/ksamart@xsede.org/integrative-drugrep-tb/data/v2/figure3/filtered_shortest_paths_bcscores_persource.csv


In [51]:
# Compute degree centrality for all nodes
degree_centrality = nx.degree_centrality(G)

# Define threshold for hubs (top 1% or 2%, adjust as needed)
threshold = sorted(degree_centrality.values(), reverse=True)[int(len(degree_centrality) * 0.2)]

# Identify hub genes
hub_genes = {node for node, centrality in degree_centrality.items() if centrality >= threshold}

print(f"Number of hub genes detected: {len(hub_genes)}")
print(hub_genes)

Number of hub genes detected: 3383
{'ZNF555', 'PNPT1', nan, 'INSRR', 'ZKSCAN1', 'KM-PA-2', 'TBC1D2B', 'UTY', 'RBM25', 'GNL2', 'DHODH', 'UBA1', 'INSR', 'MRPL27', 'TOP3B', 'CIRH1A', 'SMYD4', 'PTPDC1', 'ZNF81', 'MARK3', 'SGTA', 'ZNF679', 'TNNC2', 'MYO1H', 'HERC4', 'POLDIP3', 'HIRA', 'RPS6KA6', 'PUS7', 'HIST1H2AJ', 'SOS1', 'PCBP2', 'CNTD2', 'TEX10', 'CDK4', 'GRPEL2', 'ABT1', 'UPF1', 'ZNF286A', 'PRKCB', 'ZNF556', 'JUND', 'MRPS16', 'RGPD5', 'RHOQ', 'ACACB', 'ZNF737', 'KDM6B', 'ARL17A', 'DIS3L', 'CARS2', 'YWHAE', 'NOB1', 'TXN2', 'MTF1', 'DUS1L', 'RPL17', 'KLF10', 'PGK2', 'PIK3CD', 'PRPF19', 'RAB28', 'RECQL', 'IQCB1', 'GNL1', 'RPL28', 'UCKL1', 'SF3A2', 'CDK8', 'PPIG', 'ZNF460', 'MAPRE3', 'C12orf10', 'NOTCH1', 'MPND', 'ZNF705D', 'RACGAP1', 'TSG101', 'MFN2', 'VPS72', 'ZNF571', 'ZBTB43', 'ANKRD45', 'SIK1', 'KHDRBS1', 'STAT1', 'STXBP3', 'PPP3CA', 'REM1', 'ZNF791', 'ZNF740', 'ZNF668', 'RPS19', 'LGR4', 'ZNF142', 'ZNF135', 'GNAI3', 'RPL38', 'DNM3', 'FARSA', 'RSL1D1', 'HMG20A', 'ZNF157', 'MCMDC2', 'AC

In [52]:
top_paths_per_source

{'PARP1': {('PARP1', 'HSPA8'): ['PARP1', 'UBC', 'HSPA8'],
  ('PARP1', 'PCNA'): ['PARP1', 'UBC', 'PCNA'],
  ('PARP1', 'CCNB1'): ['PARP1', 'UBC', 'CCNB1'],
  ('PARP1', 'RHOA'): ['PARP1', 'UBC', 'RHOA'],
  ('PARP1', 'CDKN1A'): ['PARP1', 'UBC', 'CDKN1A'],
  ('PARP1', 'APP'): ['PARP1', 'UBC', 'APP'],
  ('PARP1', 'HMGCR'): ['PARP1', 'UBC', 'HMGCR']},
 'ADRB2': {('ADRB2', 'HSPA8'): ['ADRB2', 'UBC', 'HSPA8'],
  ('ADRB2', 'PCNA'): ['ADRB2', 'UBC', 'PCNA'],
  ('ADRB2', 'CCNB1'): ['ADRB2', 'UBC', 'CCNB1'],
  ('ADRB2', 'RHOA'): ['ADRB2', 'UBC', 'RHOA'],
  ('ADRB2', 'CDKN1A'): ['ADRB2', 'UBC', 'CDKN1A'],
  ('ADRB2', 'APP'): ['ADRB2', 'UBC', 'APP'],
  ('ADRB2', 'HMGCR'): ['ADRB2', 'UBC', 'HMGCR']},
 'ALAS1': {('ALAS1', 'HSPA8'): ['ALAS1', 'UBC', 'HSPA8'],
  ('ALAS1', 'PCNA'): ['ALAS1', 'UBC', 'PCNA'],
  ('ALAS1', 'CCNB1'): ['ALAS1', 'UBC', 'CCNB1'],
  ('ALAS1', 'RHOA'): ['ALAS1', 'UBC', 'RHOA'],
  ('ALAS1', 'CDKN1A'): ['ALAS1', 'UBC', 'CDKN1A'],
  ('ALAS1', 'APP'): ['ALAS1', 'UBC', 'APP'],
  ('ALAS1

In [55]:
import pandas as pd

edges = []

for source, path_dict in top_paths_per_source.items():
    for key, path in path_dict.items():
        # Generate consecutive node pairs from each path
        for i in range(len(path) - 1):
            edge = (path[i], path[i+1])
            edges.append(edge)

edge_table_df = pd.DataFrame(edges, columns=['Source', 'Target'])




edge_table_df['Interaction'] = 'interacts_with'

# 3. Save as CSV (ready for Cytoscape)
edge_table_df.to_csv("edge_table.csv", index=False)

print(edge_table_df)

    Source Target     Interaction
0    PARP1    UBC  interacts_with
1      UBC  HSPA8  interacts_with
2    PARP1    UBC  interacts_with
3      UBC   PCNA  interacts_with
4    PARP1    UBC  interacts_with
..     ...    ...             ...
331    UBC  HMGCR  interacts_with
332  UBE2C    UBC  interacts_with
333    UBC  HIF1A  interacts_with
334  UBE2C    UBC  interacts_with
335    UBC  ADRB2  interacts_with

[336 rows x 3 columns]


In [53]:
# Extract all unique genes (nodes) between sources and targets
intermediate_genes = set()

for source, paths in top_paths_per_source.items():
    for key, path in paths.items():
        if path:  # Ensure the path is not None
            # Exclude source and target nodes
            intermediate_genes.update(path[1:-1])
print(f"Number of unique intermediate genes: {len(intermediate_genes)}")
print(intermediate_genes)

intermediate_genes = {gene for gene in intermediate_genes if gene not in hub_genes}

print(f"Number of unique intermediate genes: {len(intermediate_genes)}")
print("Genes between sources and targets:")
print(intermediate_genes)


Number of unique intermediate genes: 15
{'CBX5', 'PRPF6', 'TRIM28', 'CDKN1A', 'CD2BP2', 'CDK5', 'CANX', 'CD2', 'UBC', 'PRPF8', 'BMPR1A', 'IRS1', 'INSR', 'CCND2', 'TP53'}
Number of unique intermediate genes: 8
Genes between sources and targets:
{'CBX5', 'TRIM28', 'CDKN1A', 'CD2BP2', 'CD2', 'BMPR1A', 'IRS1', 'CCND2'}


#### Create edge table for network viz

In [58]:
import pandas as pd

# Load your drug-gene table
drug_gene_df = pd.read_csv("/projects/ksamart@xsede.org/integrative-drugrep-tb/data/v2/figure3/drug_gene_table.tsv", sep="\t")  # Update path if needed

# Your full paths_dict
paths_dict = top_paths_per_source

# Step 1: Extract all unique genes from paths_dict
def extract_genes_from_paths(paths_dict):
    all_genes = set()
    for paths in paths_dict.values():
        for path in paths.values():
            all_genes.update(path)
    return all_genes

# Step 2: Build unique undirected gene–gene edges
def build_unique_gene_gene_edges(paths_dict):
    edge_set = set()
    for paths in paths_dict.values():
        for path in paths.values():
            for i in range(len(path) - 1):
                node1, node2 = path[i], path[i+1]
                edge = tuple(sorted([node1, node2]))  # Sort for undirected equivalence
                edge_set.add(edge)
    # Convert to edge list with interaction type
    return [(edge[0], edge[1], "gene-gene") for edge in edge_set]

# Generate unique undirected gene–gene edges
gene_gene_edges = build_unique_gene_gene_edges(paths_dict)

# Build filtered drug–gene edges (keep as directed)
drug_gene_edges = build_filtered_drug_gene_edges(drug_gene_df, genes_in_paths)

# Combine all edges
all_edges = gene_gene_edges + drug_gene_edges
edge_table_df = pd.DataFrame(all_edges, columns=["Source", "Target", "Interaction"])
edge_table_df = edge_table_df.drop_duplicates()

# Save edge table
edge_table_df.to_csv("cytoscape_edge_table.csv", index=False)
edge_table_df

,Source,Target,Interaction
0,CCND2,CDK5,gene-gene
1,TRIM28,UBC,gene-gene
2,CBX5,SUV39H1,gene-gene
3,EGR1,TP53,gene-gene
4,CD2BP2,PRPF6,gene-gene
...,...,...,...
85,dopamine,SUV39H1,drug-targets
86,cabergoline,CCNB1,drug-targets
87,aniracetam,CD58,drug-targets
88,propafenone,EDEM1,drug-targets


#### Create node table for network viz

In [75]:
# Step 1: Nodes from paths_dict (default type: 'gene')
def build_nodes_from_paths(paths_dict):
    all_genes = set()
    for paths in paths_dict.values():
        for path in paths.values():
            all_genes.update(path)
    return pd.DataFrame({'node': list(all_genes), 'type': 'gene'})

# Step 2: Genes from drug_gene_df using their own 'type'
def build_gene_nodes_with_types(drug_gene_df):
    gene_nodes_df = drug_gene_df[['drug_gene', 'type']].drop_duplicates()
    gene_nodes_df = gene_nodes_df.rename(columns={'drug_gene': 'node'})
    return gene_nodes_df

# Step 3: Drug nodes
def build_drug_nodes(drug_gene_df):
    unique_drugs = drug_gene_df['drug_name'].unique()
    return pd.DataFrame({'node': unique_drugs, 'type': 'drug'})

# Step 4: Combine all nodes
def build_node_table(paths_dict, drug_gene_df):
    genes_from_paths_df = build_nodes_from_paths(paths_dict)
    genes_with_types_df = build_gene_nodes_with_types(drug_gene_df)
    drugs_df = build_drug_nodes(drug_gene_df)

    # Use gene-specific type where available; fallback to 'gene'
    combined_genes_df = pd.concat([genes_with_types_df, genes_from_paths_df], ignore_index=True)
    combined_genes_df = combined_genes_df.drop_duplicates(subset='node', keep='first')

    # Combine with drugs
    node_table_df = pd.concat([combined_genes_df, drugs_df], ignore_index=True)
    node_table_df = node_table_df.drop_duplicates(subset='node')

    return node_table_df

# Build node table
node_table_df = build_node_table(paths_dict, drug_gene_df)

def classify_node(row):
    if row['node'] in sources:
        return 'source'
    elif row['node'] in intermediate_genes:
        return 'in-path'
    elif row['node'] in hub_genes and row['node'] not in targets:
        return 'hub'
    else:
        return row['type']


# Apply classification
node_table_df['type'] = node_table_df.apply(classify_node, axis=1)
node_table_df.to_csv("cytoscape_node_table.csv", index=False)

# Optional: View
print(node_table_df)

                  node   type
0                DNM1L  LINCS
1               KIF20A  LINCS
2                ARL4C  LINCS
3               TXNDC9  LINCS
4                 CDK6  LINCS
..                 ...    ...
519      dicycloverine   drug
520       benzthiazide   drug
521  meclofenamic-acid   drug
522        ziprasidone   drug
523       procainamide   drug

[524 rows x 2 columns]


In [71]:
len(sources)

17

In [74]:
# Count how many nodes are labeled as 'source'
print("Total nodes labeled as source:", sum(node_table_df['type'] == 'source'))


Total nodes labeled as source: 14
